# Patch-clamp: analysis of passive membrane properties using the eFEL library

In this tutorial, I explain how to calculate basic passive membrane properties from current clamp recordings using the [eFel library](https://github.com/BlueBrain/eFEL). To read the full tutorial, please see [Patch-clamp data analysis in Python: passive membrane propertiesPatch-clamp data analysis in Python: passive membrane properties](https://spikesandbursts.wordpress.com/2022/05/13/patch-clamp-data-analysis-python-passive-membrane-properties/) of the [Spikes and Bursts](https://spikesandbursts.wordpress.com/) blog.

The [Electrophys Feature Extract Library (eFEL)](https://github.com/BlueBrain/eFEL) is a library written in Python by the Blue Brain Project to automatically extract features from electrophysiological recordings. 
* Documentation can be found here: https://efel.readthedocs.io/en/latest/
* Description of electrophysiological features is available here: https://efel.readthedocs.io/en/latest/eFeatures.html

To load the ABF files, use the pyABF package https://swharden.com/pyabf/https://swharden.com/pyabf/

All external libraries in this notebook comply with their respective licenses, and full credit is given to their original authors and contributors.

# Import the libraries

In [ ]:
# Import the packages 
import pyabf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import efel

# List of available features in the eFEL package
# efel.getFeatureNames() 

# Define the paths to folders and files

In [ ]:
notebook_name = 'passive_membrane_properties_efel'

# Data path to 'Data_example' folders. Change accordingly to your data structure.
data_path = os.path.dirname(os.getcwd())  # Moves one level up from the current directory

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Load the example file

The example data for this notebook is the file **pfc_pyr_aps_03.abf**.

In [ ]:
# ABF file/s
filename = "pfc_pyr_aps_03"

data_path = f"{paths['data']}/{filename}.abf" 
abf = pyabf.ABF(data_path)
print(abf)

# Quick plot to see the trace/s
plt.figure(figsize=(8,4))

for sweepNumber in abf.sweepList:
    abf.setSweep(sweepNumber)
    plt.plot(abf.sweepX, abf.sweepY)
    plt.ylabel(abf.sweepLabelY)
    plt.xlabel(abf.sweepLabelX)

plt.show()

# eFEL: passive membrane properties

Description of the electrophysiological features: https://efel.readthedocs.io/en/latest/eFeatures.html

In [ ]:
# It is convenient to set all the parameters at the beginning
sweeps = abf.sweepList[0:12]                 # Remove the square brackets to select all sweeps
stim_start = 230                             # Beginning of the current stimulus in milliseconds
stim_end = 730                               # End of the current stimulus in milliseconds
sampling_freq = 10000                        # Sampling frequency (Hz)
currentstep_time = int(0.4 * sampling_freq)  # Time point (s) within a current step

# Define the range of current steps to calculate input resistance
ir_current_begin = -80            # Current step (pA)
ir_current_end = 0                # Current step (pA)
ir_index_begin = 8                # Index (position) of the current step
ir_index_end = 12                 # Index (position) of the current step

# Create the output table
table = pd.DataFrame(columns=[
    'voltage',
    'steady_state_voltage_stimend',
    'voltage_delta',
    'current_pA',
    'input_resistance_Gohm',
    'time_constant',
    'capacitance_pF',
    'sag_amplitude',
    'sag_ratio1'
])

# Loop through sweeps
for sweep in sweeps:
    abf.setSweep(sweep)

    # Define trace and region of analysis
    trace = {
        'T': abf.sweepX * 1000,  # Convert to ms
        'V': abf.sweepY,
        'I': abf.sweepC,
        'stim_start': [stim_start],
        'stim_end': [stim_end]}
    traces = [trace]

    # Optional: adjust parameters for the time constant calculation
    efel.api.setDoubleSetting('decay_start_after_stim', 0)   # default = 1 ms
    efel.api.setDoubleSetting('decay_end_after_stim', 30)    # default = 10 ms

    # Extract features
    feature_values = efel.getFeatureValues(
        traces,
        ['voltage',
         'steady_state_voltage_stimend',
         'current',
         'decay_time_constant_after_stim',
         'sag_amplitude',
         'sag_ratio1'],
        raise_warnings=None)[0]

    # Append results to table
    idx = len(table)
    table.loc[idx, 'voltage'] = feature_values['voltage'][0]
    table.loc[idx, 'steady_state_voltage_stimend'] = feature_values['steady_state_voltage_stimend'][0]
    table.loc[idx, 'voltage_delta'] = (
        feature_values['steady_state_voltage_stimend'] - feature_values['voltage']
    )[0]
    table.loc[idx, 'current_pA'] = feature_values['current'][currentstep_time]
    table.loc[idx, 'sag_amplitude'] = (feature_values['sag_amplitude'][0] if feature_values['sag_amplitude'] is not None else np.nan)
    table.loc[idx, 'sag_ratio1'] = (feature_values['sag_ratio1'][0] if feature_values['sag_amplitude'] is not None else np.nan)

    # Input resistance and capacitance
    if ir_current_begin <= table.loc[idx, 'current_pA'] < ir_current_end:
        table.loc[idx, 'input_resistance_Gohm'] = (
            table.loc[idx, 'voltage_delta'] / table.loc[idx, 'current_pA']
        )
        if feature_values['decay_time_constant_after_stim'] is not None:
            tau = feature_values['decay_time_constant_after_stim'][0]
            R = table.loc[idx, 'input_resistance_Gohm']
            table.loc[idx, 'time_constant'] = tau
            table.loc[idx, 'capacitance_pF'] = tau / R

# Plotting
fig = plt.figure(figsize=(12, 4))
fig.tight_layout()

# I–V curve to estimate the input resistance
ax1 = fig.add_subplot(1, 2, 1)
currents = []
for sweep in sweeps:
    abf.setSweep(sweep)
    currents.append(abf.sweepC[currentstep_time])
x = currents

# Get steady-state voltage at the end of each current step
y = table['steady_state_voltage_stimend']
y_list = y.to_list()

# Select range of current steps used to calculate input resistance
x_ir = x[ir_index_begin:ir_index_end]
y_ir = y_list[ir_index_begin:ir_index_end]

# Linear regression: slope = input resistance, intercept = baseline voltage
m, b = np.polyfit(x_ir, y_ir, 1)

# Plot the regression line and raw data
line = np.polyval([m, b], x_ir)
ax1.plot(x_ir, line, color='magenta', label='Linear fit')
ax1.scatter(x, y, color='gray', label='Data')
ax1.set_xlabel('Current (pA)')
ax1.set_ylabel('Voltage (mV)')
ax1.set_title('I–V curve')
ax1.legend()

# Voltage traces
ax2 = fig.add_subplot(2, 2, 2)
for sweep in sweeps:
    abf.setSweep(sweep)
    ax2.plot(abf.sweepX * 1000, abf.sweepY, alpha=0.3)
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Membrane voltage (mV)')
ax2.set_xlim(0, 1500)

# Current steps
ax3 = fig.add_subplot(2, 2, 4, sharex=ax2)
for sweep in sweeps:
    abf.setSweep(sweep)
    ax3.plot(abf.sweepX * 1000, abf.sweepC, alpha=0.3)
ax3.set_xlabel('Time (ms)')
ax3.set_ylabel('Current (pA)')
ax3.set_xlim(0, 1500)

# Save results
fig.savefig(f"{paths['analysis']}/{filename}.png", dpi=300)
table.to_csv(f"{paths['analysis']}/{filename}_passive_memb_properties.csv", index=False)

# Display
print("Input resistance (GΩ) =", m)
plt.show()
table